# N7 — Error analysis and OOD evaluation

In [ ]:
from pathlib import Path
import sys, yaml, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
CFG=yaml.safe_load((ROOT/'config.yaml').read_text())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE,'debug:',CFG['data']['debug'])

from collections import Counter

Chord-density buckets: sparse (0–2 simultaneous notes), medium (3–5), dense (6+). Apply the same bins to every system.

In [ ]:
def chord_density(roll):return np.asarray(roll).sum(-1)
def bucket(d):return 'sparse' if d<=2 else ('medium' if d<=5 else 'dense')
def classify_errors(pred,ref,tol=.08):
 remaining=list(ref);out={'matched':0,'ghost':0,'octave':0}
 for p in pred:
  i=next((i for i,r in enumerate(remaining) if p['pitch']==r['pitch'] and abs(p['onset']-r['onset'])<=tol),None)
  if i is not None:out['matched']+=1;remaining.pop(i);continue
  j=next((i for i,r in enumerate(remaining) if abs(p['pitch']-r['pitch'])==12 and abs(p['onset']-r['onset'])<=tol),None)
  if j is not None:out['octave']+=1;remaining.pop(j)
  else:out['ghost']+=1
 out['missed']=len(remaining);return out

Out-of-domain: evaluate own recordings separately. Without aligned reference MIDI, report qualitative listening/visual review only, not F1. Record piano, room, microphone, sample rate and compression conditions.